In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('C:\\Users\\iProg\\OneDrive\\Documents\\Football_predict\\nfl_prediction_system\\NFL_ML_Predictions\\backend\\models\\validation_errors.csv')


In [4]:
# REQUIRED COLUMNS in df:
# 'home_team', 'away_team', and a boolean 'home_win' (True if home won, False if away won).
# Also make sure df is chronological (by game_date OR season/week).

# 1) Sort chronologically to make cumulative counts meaningful
sort_cols = [c for c in ['game_date', 'season', 'week'] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)

# 2) Canonical pair key (order-agnostic: "NE::SEA" == "SEA::NE")
df['pair_key'] = df.apply(lambda r: '::'.join(sorted([r['home_team'], r['away_team']])), axis=1)

# 3) Win flags for each side this game
df['home_won'] = (df['home_win'] == True).astype(int)
df['away_won'] = (df['home_win'] == False).astype(int)
# If ties exist, neither flag will be 1 for that row.

# 4) CUMULATIVE (shifted) wins for each side *before* this game vs the same opponent
df['pre_wins_home_vs_away'] = (
    df.groupby(['pair_key', 'home_team'])['home_won']
      .cumsum()
      .shift(fill_value=0)
)

df['pre_wins_away_vs_home'] = (
    df.groupby(['pair_key', 'away_team'])['away_won']
      .cumsum()
      .shift(fill_value=0)
)

# 5) Dominance view for the NEXT matchup:
# We present "+X TEAM_A | -Y TEAM_B" where X and Y are the pregame head-to-head win counts.
pre_home = df['pre_wins_home_vs_away']
pre_away = df['pre_wins_away_vs_home']

# Team with more prior wins gets the +, the other gets the - (ties both show +0/-0)
df['dom_plus_team']  = np.where(pre_home >= pre_away, df['home_team'], df['away_team'])
df['dom_plus_value'] = np.maximum(pre_home, pre_away)

df['dom_minus_team']  = np.where(pre_home >= pre_away, df['away_team'], df['home_team'])
df['dom_minus_value'] = -np.minimum(pre_home, pre_away)

# Pretty label for dashboards/logs
df['dominance_label'] = (
    '+' + df['dom_plus_value'].astype(int).astype(str) + ' ' + df['dom_plus_team'] +
    ' | ' + df['dom_minus_value'].astype(int).astype(str) + ' ' + df['dom_minus_team']
)

# 6) (Optional) A simple dominance *difference* metric (who leads & by how much)
df['dom_diff'] = pre_home - pre_away
# >0 => home side currently leads the all-time series before this game, <0 => away side leads.
print(df[['home_team', 'away_team', 'pre_wins_home_vs_away', 'pre_wins_away_vs_home', 'dominance_label', 'dom_diff']].head(20))
# -- Example read:
# If SEA has 8 wins and NE has 3 prior to the next game:
#   dominance_label -> "+8 SEA | -3 NE"
#   dom_diff        -> 8 - 3 = +5


   home_team away_team  pre_wins_home_vs_away  pre_wins_away_vs_home  \
0         NO       LAR                      0                      0   
1        DEN        LV                      1                      0   
2         NE       NYJ                      0                      1   
3        IND       PHI                      1                      0   
4        BAL       CAR                      0                      1   
5        BUF       CLE                      1                      0   
6        ARI        SF                      1                      0   
7        DET       BUF                      0                      1   
8        MIN        NE                      0                      1   
9        DAL       NYG                      1                      0   
10       WAS       ATL                      1                      0   
11       JAX       BAL                      1                      0   
12       NYJ       CHI                      1                   

In [3]:
# Analyze team-level prediction accuracy and dominance(e.g. how many times 1 unique team beats another unique team)
# # e.g. ari vs sea played 10 times, ari won 7 times, sea won 3 times - +7ari dominates -3sea etc. 

for hteam in df['home_team'].unique():
    team_df = df[(df['home_team'] == hteam) | (df['away_team'] == hteam)]

    df[hteam + 'Dominates'] = np.where(
        (df['home_team'] == hteam) and (df['prob_home_win'] > 0.7) |
        (df['away_team'] == hteam) & (df['prob_home_win'] < 0.3),
        True, False
    )
    accuracy = ((team_df['prob_home_win'] > 0.5) == team_df['home_win']).mean()
    print(f"Team: {hteam}, Prediction Accuracy: {accuracy:.2%}")

total_predictions = len(df)
correct_predictions = 0
incorrect_predictions = 0



ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:

for i in range(len(df['home_win'])):
    if df['prob_home_win'][i] > 0.5 and df['home_win'][i] == 1:
        print("Correct Prediction")
        print("Prediction Confidence", {df['home_team'][i]}, {df['prob_home_win'][i]})
        correct_predictions += 1

        
    else:
        print("Wrong Prediction")
        print("Prediction Confidence", {df['home_team'][i]}, {df['prob_home_win'][i]})
